# 03 — Exploration des embeddings BGE-M3

**Objectif** : comprendre concrètement ce que fait l'encodeur.

Ce que le notebook vérifie, dans l'ordre :

1. l'environnement GPU ;
2. la révision du modèle à épingler ;
3. la composition du modèle sentence-transformers (Transformer → Pooling → Normalize) ;
4. la tokenisation et les tokens spéciaux ;
5. le **pooling CLS refait à la main**, comparé au mean pooling ;
6. la similarité sur des phrases synthétiques ;
7. l'encodage des vrais chunks (forme, normes, durée, mémoire GPU) ;
8. l'écart entre fp16 et fp32 ;
9. les plus proches voisins entre chunks (codes seulement).


In [10]:
import time

import numpy as np
import torch
import torch.nn.functional as F
from huggingface_hub import model_info
from sentence_transformers import SentenceTransformer

from assistant_regles.ingest.chunk import lire_jsonl
from assistant_regles.ingest.config import charger_config

## 1. Environnement

In [ ]:
print("torch", torch.__version__, "| CUDA", torch.version.cuda)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device :", device, "|", torch.cuda.get_device_name(0) if device == "cuda" else "CPU")

torch 2.14.0+cu130 | CUDA 13.0
device : cuda | NVIDIA GeForce RTX 5070 Ti


## 2. Révision du modèle à épingler

Un dépôt Hugging Face évolue : `BAAI/bge-m3` sans révision désigne « la dernière version ». On récupère le hash du commit actuel. C'est cette valeur qui ira dans `config/rag/indexation.yaml`, pour que les vecteurs soient reproductibles.

In [ ]:
NOM_MODELE = "BAAI/bge-m3"
REVISION = model_info(NOM_MODELE).sha
print(REVISION)

5617a9f61b028005a4858fdac845db406aefb181


## 3. Chargement et composition du modèle

Le premier chargement télécharge environ 2,3 Go dans le cache Hugging Face (`~/.cache/huggingface`), hors du repo.

On passe le modèle en **fp16** sur GPU : deux fois moins de mémoire et plus rapide. La section 8 mesure ce que ça coûte en précision.

À lire dans l'affichage du modèle : les trois modules empilés, et en particulier la ligne `Pooling`, qui doit indiquer le mode **CLS**.

In [ ]:
modele = SentenceTransformer(NOM_MODELE, revision=REVISION, device=device)
if device == "cuda":
    modele.half()

print(modele)
print("dimension     :", modele.get_embedding_dimension())
print("longueur max  :", modele.max_seq_length, "tokens")

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 58418.15it/s]


SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'XLMRobertaModel'})
  (1): Pooling({'embedding_dimension': 1024, 'pooling_mode': 'cls', 'include_prompt': True})
  (2): Normalize({'module_input_name': 'sentence_embedding', 'module_output_name': 'sentence_embedding'})
)
dimension     : 1024
longueur max  : 8192 tokens


## 4. Tokenisation : les tokens spéciaux

Observe le premier et le dernier token : `<s>` et `</s>`. Ils sont ajoutés par le tokenizer et ne sont pas comptés dans `nb_tokens` à l'ingestion (d'où le « + 2 » du contrôle de longueur). Le préfixe `▁` marque un début de mot (tokenizer SentencePiece).

In [8]:
phrase = "Une unité qui a avancé ne peut pas tirer, sauf règle contraire."

ids = modele.tokenizer(phrase)["input_ids"]
print(len(ids), "tokens")
print(ids)
print(modele.tokenizer.convert_ids_to_tokens(ids))

18 tokens
[0, 8975, 51, 5097, 569, 10, 193193, 108, 4372, 452, 18657, 56, 4, 138477, 202867, 119915, 5, 2]
['<s>', '▁Une', '▁un', 'ité', '▁qui', '▁a', '▁avancé', '▁ne', '▁peut', '▁pas', '▁tir', 'er', ',', '▁sauf', '▁règle', '▁contraire', '.', '</s>']


## 5. Le pooling CLS refait à la main

On appelle directement le transformer (`modele[0].auto_model`) pour récupérer la matrice brute « un vecteur par token », puis on la résume de deux façons :

- **CLS** : le vecteur du premier token (`<s>`) ;
- **moyenne** : la moyenne des vecteurs de tous les tokens (pondérée par le masque d'attention).

On compare chacun à la sortie officielle de `modele.encode`. Attendu : CLS ≈ 1,000 (aux arrondis fp16 près) et la moyenne nettement en dessous. Ça prouve que sentence-transformers applique le bon pooling, et que se tromper de pooling change réellement le vecteur.

In [11]:
entree = modele.tokenizer([phrase], return_tensors="pt").to(modele.device)

In [16]:
entree

{'input_ids': tensor([[     0,   8975,     51,   5097,    569,     10, 193193,    108,   4372,
            452,  18657,     56,      4, 138477, 202867, 119915,      5,      2]],
       device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]],
       device='cuda:0')}

In [17]:
with torch.no_grad():
    etats = modele[0].auto_model(**entree).last_hidden_state   # (1, n_tokens, 1024)
print("matrice brute :", tuple(etats.shape))

matrice brute : (1, 18, 1024)


In [20]:
cls = F.normalize(etats[:, 0].float(), dim=-1)
cls.shape

torch.Size([1, 1024])

In [21]:
masque = entree["attention_mask"].unsqueeze(-1).float()
moyenne = F.normalize((etats.float() * masque).sum(1) / masque.sum(1), dim=-1)
moyenne.shape

torch.Size([1, 1024])

In [22]:
officiel = modele.encode([phrase], convert_to_tensor=True, normalize_embeddings=True).float()

print(f"cosinus CLS manuel / encode     : {(cls @ officiel.T).item():.4f}")
print(f"cosinus moyenne manuelle / encode : {(moyenne @ officiel.T).item():.4f}")

cosinus CLS manuel / encode     : 1.0000
cosinus moyenne manuelle / encode : 0.7699


## 6. Similarités sur des phrases synthétiques

Les vecteurs étant normalisés, le produit scalaire **est** la similarité cosinus. Attendu : la question (0) est plus proche de la règle pertinente (1) que de la règle voisine (2), elle-même plus proche que le hors-sujet (3).

**Piège d'interprétation** : les valeurs absolues sont souvent élevées, même pour des textes sans rapport. Les vecteurs d'un modèle occupent une zone restreinte de l'espace (on parle d'anisotropie). Ce qui compte, c'est **l'ordre** des similarités, pas un seuil absolu.

In [25]:
phrases = [
    "Combien de pouces une unité peut-elle parcourir pendant son mouvement ?",           # 0 question
    "Une unité se déplace d'une distance au plus égale à sa caractéristique de Mouvement.",  # 1 règle pertinente
    "Une figurine blessée doit effectuer un jet de sauvegarde.",                        # 2 règle voisine
    "Faites fondre le beurre avant d'ajouter la farine.",                               # 3 hors sujet
]
vecteurs = modele.encode(phrases, normalize_embeddings=True)
print("type renvoyé :", vecteurs.dtype, vecteurs.shape)

vecteurs = vecteurs.astype(np.float32)
product = np.round(vecteurs @ vecteurs.T, 3)
print(product.shape)
print(product)


type renvoyé : float16 (4, 1024)
(4, 4)
[[1.    0.641 0.452 0.245]
 [0.641 1.001 0.49  0.367]
 [0.452 0.49  1.    0.425]
 [0.245 0.367 0.425 1.   ]]


## 7. Encodage des vrais chunks

On charge les chunks avec les outils de l'ingestion (même config, même lecteur typé), puis on vérifie :

- qu'aucun chunk ne dépasse la longueur max (sinon troncature **silencieuse**) ;
- la forme, le type, les normes et l'absence de NaN ;
- la durée et le pic de mémoire GPU, qui aideront à choisir `batch_size`.

Note : en fp16, `encode` renvoie du `float16`. On convertit en `float32`, le format de stockage de pgvector.

In [31]:
config = charger_config()
chunks = lire_jsonl(config.chemin_chunks)
textes = [c.texte for c in chunks]
print(f"Nombre de chunks a encoder : {len(textes)}")

Nombre de chunks a encoder : 207


In [32]:
max_tokens = max(c.nb_tokens for c in chunks)
print(len(chunks), "chunks | nb_tokens max :", max_tokens)
assert max_tokens + 2 <= modele.max_seq_length, "un chunk serait tronqué"

207 chunks | nb_tokens max : 568


In [33]:
if device == "cuda":
    torch.cuda.reset_peak_memory_stats()
debut = time.perf_counter()
emb = modele.encode(textes, batch_size=32, normalize_embeddings=True, show_progress_bar=True)
duree = time.perf_counter() - debut

Batches: 100%|██████████| 7/7 [00:00<00:00, 12.26it/s]


In [34]:
print("type brut :", emb.dtype)
emb = emb.astype(np.float32)
normes = np.linalg.norm(emb, axis=1)
print("forme :", emb.shape, "| durée :", f"{duree:.1f} s")
print(f"normes : min {normes.min():.5f}  max {normes.max():.5f}")
print("NaN :", bool(np.isnan(emb).any()))
if device == "cuda":
    print(f"pic VRAM : {torch.cuda.max_memory_allocated() / 1e9:.2f} Go")

type brut : float16
forme : (207, 1024) | durée : 0.6 s
normes : min 0.99976  max 1.00046
NaN : False
pic VRAM : 2.73 Go


## 8. fp16 contre fp32

On réencode les mêmes chunks avec un modèle en pleine précision (fp32) et on mesure le cosinus entre les deux versions de chaque vecteur. Si le minimum reste très proche de 1, le fp16 ne dégrade rien d'utile et on le garde dans la config.

In [35]:
modele32 = SentenceTransformer(NOM_MODELE, revision=REVISION, device=device)
emb32 = modele32.encode(textes, batch_size=32, normalize_embeddings=True).astype(np.float32)

cos = (emb * emb32).sum(axis=1)
print(f"cosinus fp16/fp32 : min {cos.min():.6f}  moyenne {cos.mean():.6f}")

del modele32
if device == "cuda":
    torch.cuda.empty_cache()

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 61813.46it/s]


cosinus fp16/fp32 : min 0.999726  moyenne 1.000061


## 9. Plus proches voisins entre chunks

Pour chaque chunk (un sur vingt), on cherche le chunk le plus similaire **autre que lui-même**. On n'affiche que les codes « XX.YY ».

À observer : les voisins appartiennent-ils souvent à la même section (même préfixe XX) ? C'est un premier signe, qualitatif, que l'espace vectoriel reflète la structure des règles. Ce n'est pas encore une évaluation : celle-ci viendra avec le gold set.

In [36]:
sim = emb @ emb.T
np.fill_diagonal(sim, -1.0)          # exclut le chunk lui-même
voisins = sim.argmax(axis=1)

for i in range(0, len(chunks), 20):
    j = voisins[i]
    print(f"{chunks[i].code or '(sans code)':>12} -> {chunks[j].code or '(sans code)':<12} {sim[i, j]:.3f}")

 (sans code) -> 01.01        0.717
       03.01 -> 03.04        0.813
 (sans code) -> (sans code)  0.678
       09.06 -> 09.05        0.861
       12.03 -> 12.06        0.830
       13.10 -> 13.11        0.795
 (sans code) -> (sans code)  0.680
 (sans code) -> 20.03        0.826
 (sans code) -> 24.01        0.711
       24.18 -> 24.33        0.797
 (sans code) -> (sans code)  0.786


## Bilan : ce qu'on reporte dans `rag/embeddings.py`

À noter après exécution :

- la **révision** (section 2), pour `config/rag/indexation.yaml` ;
- la **dimension** et la **longueur max** confirmées (section 3) ;
- le **pooling CLS** vérifié (section 5), qui deviendra une assertion du test marqué `modele` ;
- la **durée**, le **pic VRAM** et un `batch_size` raisonnable (section 7) ;
- la décision **fp16 oui/non** (section 8).